In [4]:
import os, json, re, argparse, random
from itertools import combinations
from typing import List, Dict, Tuple, Optional

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM


# ============================================================
# CONFIG
# ============================================================
MODEL_NAME   = "Qwen/Qwen2.5-3B-Instruct"
TRAIN_JSON   = "train_data/subtask 1/train_data.json"
TEST_JSON_ST2 = "test_data/subtask 2/test_data_subtask_2.json"
TEST_JSON_ST4 = "test_data/subtask 4/test_data_subtask_4.json"
OUT_DIR      = "subtask2_results_retrieve_first"
SEED         = 42
MAX_LEN      = 512
ICL_SHOTS    = 4
YES_TEXT     = " yes"
NO_TEXT      = " no"
TOP_K_PAIRS  = 3


# ============================================================
# TERM EXTRACTION & MATCHING
# ============================================================
CONCLUSION_MARKERS = [
    'therefore, it must be the case that', 'therefore, it can be said that',
    'therefore, we can say that a portion of', 'therefore, we can say that',
    'from this, it follows that', 'it is necessarily concluded that',
    'it is necessarily true that', 'one can thus conclude that',
    'it is logically necessary that', 'it is therefore the case that',
    'this leads to the conclusion that', 'it must be true that',
    'it is therefore true that', 'it must follow that',
    'it can be deduced that', 'it can be said that',
    'it must be the case that', 'we can conclude that',
    'it is deduced that', 'it follows directly that',
    'it follows that', 'this implies that',
    'we can say that', 'this means that',
    'the conclusion is that', 'there exists at least',
    'consequently, some of the', 'consequently, there are no',
    'therefore, at least one',
    'consequently,', 'therefore,', 'hence,', 'thus,', 'as such,',
    'at least',
]

QUANTIFIERS = [
    'everything that can be classified as a', 'everything that can be classified as',
    'anything that is a', 'anything that is an', 'anything that is',
    'nothing in the category of', 'all of those who are',
    'all things that are', 'among the items that are',
    'there is not a single', 'it is a known fact that',
    'it is a fact that', 'it is true that',
    'it is undeniable that', 'it is the case that',
    'it is known that', 'a certain number of',
    'a number of', 'a portion of',
    'there are no', 'there are some', 'there are many',
    'there exist some', 'there exist', 'there exists',
    'every single', 'without exception', 'without a doubt',
    'every', 'each', 'all', 'some', 'any', 'no',
    'not all', 'a few', 'in fact', 'of course',
]

STOPS = frozenset({
    'is','are','was','were','be','been','being','has','have','had',
    'do','does','did','will','would','shall','should','may','might',
    'can','could','the','a','an','of','in','to','for','with',
    'by','on','at','from','as','into','that','which','who','whom',
    'this','these','those','it','its','also','and','or','but','if',
    'than','when','where','how','what','there','here','only',
    'defined','described','classified','considered','called',
    'type','kind','form','actually','fact','known','true','single',
    'said','necessarily','must','follows','therefore','consequently',
    'hence','such','means','case','leads','conclusion','deduced',
    'established','never','mutually','exclusive','one','not',
    'certain','number','least','thus','we','conclude','can',
    'portion','directly','exists','logically','necessary',
    'course','without','exception','doubt','undeniable',
    'thing','things','item','items','upon','about','over',
})


def normalize_word(word: str) -> str:
    w = word.lower().strip('.,!?;:()')
    if w.endswith('ies') and len(w) > 4:
        return w[:-3] + 'y'
    if w.endswith('es') and len(w) > 4 and w[-3] not in 'aeiou':
        return w[:-2]
    if w.endswith('s') and not w.endswith('ss') and len(w) > 3:
        return w[:-1]
    return w


def extract_terms(sentence: str) -> set:
    s = sentence.lower().strip()
    for m in sorted(CONCLUSION_MARKERS, key=len, reverse=True):
        s = s.replace(m, ' ')
    for q in sorted(QUANTIFIERS, key=len, reverse=True):
        s = s.replace(q, ' ')
    words = re.findall(r'[a-z][\w-]*', s)
    terms = set()
    for w in words:
        if w not in STOPS and len(w) > 2:
            terms.add(normalize_word(w))
            terms.add(w)
    return terms


def term_overlap(t1: set, t2: set) -> set:
    overlap = t1 & t2
    for a in t1:
        for b in t2:
            if len(a) > 4 and len(b) > 4 and (a in b or b in a):
                overlap.add(min(a, b, key=len))
    return overlap


def split_syllogism(text: str) -> Tuple[List[str], str]:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    if len(sentences) < 2:
        return sentences, sentences[-1] if sentences else ""
    return sentences[:-1], sentences[-1]


# ============================================================
# PREMISE PAIR SCORING
# ============================================================
def score_premise_pairs(premises: List[str], conclusion: str) -> List[Tuple[int, int, float]]:
    conc_terms = extract_terms(conclusion)
    prem_terms = [extract_terms(p) for p in premises]

    pair_scores = []
    for i, j in combinations(range(len(premises)), 2):
        ti, tj = prem_terms[i], prem_terms[j]

        shared = term_overlap(ti, tj)
        middle = shared - term_overlap(shared, conc_terms)

        ei = term_overlap(ti, conc_terms)
        ej = term_overlap(tj, conc_terms)

        middle_score = len(middle)
        ep_i, ep_j = len(ei), len(ej)

        score = 0.0
        score += middle_score * 3
        score += ep_i + ep_j

        if ep_i > 0 and ep_j > 0 and len(ei & ej) == 0:
            score += 5
        elif ep_i > 0 and ep_j > 0:
            score += 2

        if middle_score > 0 and (ep_i > 0 or ep_j > 0):
            score += 3

        if ei == ej and len(ei) > 0:
            score -= 1

        both_overlap = ei & ej
        if len(both_overlap) > 1:
            score -= len(both_overlap) * 0.5

        useful_i = len(ei) + len(term_overlap(ti, shared))
        useful_j = len(ej) + len(term_overlap(tj, shared))
        extra_i = max(0, len(ti) - useful_i)
        extra_j = max(0, len(tj) - useful_j)
        score -= (extra_i + extra_j) * 0.3

        pair_scores.append((i, j, score))

    pair_scores.sort(key=lambda x: x[2], reverse=True)
    return pair_scores


def get_top_k_pairs(premises: List[str], conclusion: str, k: int = TOP_K_PAIRS) -> List[Tuple[int, int, float]]:
    if len(premises) < 2:
        return []
    return score_premise_pairs(premises, conclusion)[:k]


# ============================================================
# PROMPTING
# ============================================================
def build_icl_examples(train_data, shots, seed):
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"] is True]
    neg = [x for x in train_data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = shots // 2
    chosen = pos[:half] + neg[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_prompt_for_pair(tokenizer, train_data, pair_premises: List[str], conclusion: str) -> str:
    chosen = build_icl_examples(train_data, ICL_SHOTS, SEED)

    messages = [
        {"role": "system", "content": (
            "You are a strict formal logic reasoner. "
            "Decide only whether the conclusion logically follows from the given premises. "
            "Ignore plausibility and world knowledge. "
            "Reply with only yes or no."
        )}
    ]

    for ex in chosen:
        prem, conc = split_syllogism(ex["syllogism"])
        prem = prem[:2] if len(prem) >= 2 else prem
        ex_text = "\n".join([f"Premise {idx+1}: {p}" for idx, p in enumerate(prem)])
        ex_text += f"\nConclusion: {conc}"
        ans = "yes" if ex["validity"] else "no"

        messages.append({
            "role": "user",
            "content": f"Argument:\n{ex_text}\n\nAnswer yes or no."
        })
        messages.append({"role": "assistant", "content": ans})

    pair_text = "\n".join([f"Premise {idx+1}: {p}" for idx, p in enumerate(pair_premises)])
    pair_text += f"\nConclusion: {conclusion}"

    messages.append({
        "role": "user",
        "content": f"Argument:\n{pair_text}\n\nAnswer yes or no."
    })

    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def score_label(model, tokenizer, prompt, label_text, max_len=MAX_LEN):
    full = prompt + label_text
    full_ids = tokenizer(
        full,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=max_len
    ).input_ids.to(model.device)

    prompt_ids = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=max_len
    ).input_ids.to(model.device)

    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)

    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity_from_pair(model, tokenizer, train_data, pair_premises: List[str], conclusion: str) -> Tuple[bool, float]:
    prompt = build_prompt_for_pair(tokenizer, train_data, pair_premises, conclusion)
    y = score_label(model, tokenizer, prompt, YES_TEXT)
    n = score_label(model, tokenizer, prompt, NO_TEXT)
    valid = y >= n
    margin = y - n
    return valid, margin


# ============================================================
# MULTILINGUAL HELPERS
# ============================================================
def get_syllogism_text(example: dict) -> str:
    if 'syllogism_t' in example and example.get('lang', 'en') != 'en':
        return example['syllogism_t']
    return example['syllogism']


def get_english_syllogism(example: dict) -> str:
    return example['syllogism']


# ============================================================
# RETRIEVE-FIRST PIPELINE
# ============================================================
@torch.no_grad()
def run_pipeline_retrieve_first(model, tokenizer, train_data, test_data, use_translated=False, top_k=TOP_K_PAIRS):
    predictions = []

    for idx, ex in enumerate(test_data):
        english_syl = get_english_syllogism(ex)
        premises, conclusion = split_syllogism(english_syl)

        if len(premises) < 2:
            predictions.append({
                "id": ex["id"],
                "validity": False,
                "relevant_premises": []
            })
            continue

        top_pairs = get_top_k_pairs(premises, conclusion, k=top_k)

        best_valid = False
        best_margin = float("-inf")
        best_pair = None

        for i, j, pair_score in top_pairs:
            candidate_premises = [premises[i], premises[j]]
            is_valid, margin = predict_validity_from_pair(
                model, tokenizer, train_data, candidate_premises, conclusion
            )

            # small bias toward stronger heuristic chain score
            combined_margin = margin + 0.05 * pair_score

            if combined_margin > best_margin:
                best_margin = combined_margin
                best_valid = is_valid
                best_pair = sorted([i, j])

        if best_valid and best_pair is not None:
            relevant = best_pair
        else:
            relevant = []

        predictions.append({
            "id": ex["id"],
            "validity": bool(best_valid),
            "relevant_premises": relevant,
        })

        if (idx + 1) % 50 == 0:
            print(f"  {idx + 1}/{len(test_data)}")

    return predictions


# ============================================================
# MAIN
# ============================================================
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--subtask", type=int, default=2, choices=[2, 4])
    parser.add_argument("--test_json", default=None)
    parser.add_argument("--use_lora", default=None)
    parser.add_argument("--out_dir", default=OUT_DIR)
    parser.add_argument("--top_k_pairs", type=int, default=TOP_K_PAIRS)
    args = parser.parse_args()

    random.seed(SEED)
    os.makedirs(args.out_dir, exist_ok=True)

    if args.test_json is None:
        args.test_json = TEST_JSON_ST4 if args.subtask == 4 else TEST_JSON_ST2

    print(f"Subtask: {args.subtask}")
    print(f"Test file: {args.test_json}")

    with open(TRAIN_JSON, "r") as f:
        train_data = json.load(f)
    with open(args.test_json, "r") as f:
        test_data = json.load(f)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None
    )

    if args.use_lora:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, args.use_lora)

    model.eval()

    use_translated = (args.subtask == 4)

    print("Running retrieve-first pipeline...")
    predictions = run_pipeline_retrieve_first(
        model=model,
        tokenizer=tokenizer,
        train_data=train_data,
        test_data=test_data,
        use_translated=use_translated,
        top_k=args.top_k_pairs,
    )

    out_path = os.path.join(args.out_dir, f"predictions_st{args.subtask}.json")
    with open(out_path, "w") as f:
        json.dump(predictions, f, indent=2)

    print(f"Saved predictions to: {out_path}")


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--subtask {2,4}] [--test_json TEST_JSON]
                             [--use_lora USE_LORA] [--out_dir OUT_DIR]
                             [--top_k_pairs TOP_K_PAIRS]
ipykernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-39372476-49e1-478d-842f-6238be86adb1.json


SystemExit: 2

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
